In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

class NewsTradingBacktest:
    def __init__(self, news_df, price_df, holding_period=1, risk_free_rate=0.02):
        """
        Initialize the backtesting class.
        
        Parameters:
        - news_df: DataFrame containing columns ['title', 'date', 'sentiment_label']
        - price_df: DataFrame with a datetime index and a 'price' column
        - holding_period: Number of days to hold the position
        - risk_free_rate: Annualized risk-free rate for Sharpe ratio calculation
        """
        self.news_df = news_df
        self.price_df = price_df
        self.holding_period = holding_period
        self.risk_free_rate = risk_free_rate

    def calculate_returns(self):
        """Calculate daily returns and align them with news data."""
        self.price_df['log_return'] = np.log(self.price_df['price']/self.price_df['price'].shift(1))
        
        # Align news dates with price data
        self.news_df['date'] = pd.to_datetime(self.news_df['date'])
        self.price_df['date'] = self.price_df.index

        # Merge data
        merged_df = pd.merge(self.price_df, self.news_df, on='date', how='left')
        merged_df['position'] = np.nan
        merged_df['position'] = np.where(merged_df['sentiment_label'] == 1, 1, merged_df['position'])
        merged_df['position'] = np.where(merged_df['sentiment_label'] == -1, -1, merged_df['position'])
        
        # Generate position based on signal and holding period
        merged_df['position'] = merged_df['signal'].ffill(limit=self.holding_period)
        merged_df['position'] = merged_df['position'].fillna(0)
        # Calculate strategy returns
        merged_df['strategy_return'] = merged_df['position'].shift(1) * merged_df['log_return']
        
        self.results_df = merged_df.dropna()

    def plot_cumulative_returns(self):
        """Plot cumulative returns of the strategy."""
        self.results_df['cumulative_strategy'] = (self.results_df['strategy_return']).cumsum().apply(np.exp)

        plt.figure(figsize=(10, 6))
        plt.plot(self.results_df['date'], self.results_df['cumulative_strategy'], label='Strategy')
        plt.title('Cumulative Returns')
        plt.xlabel('Date')
        plt.ylabel('Cumulative Returns')
        plt.legend()
        plt.grid()
        plt.show()

    def calculate_sharpe_ratio(self):
        """Calculate the Sharpe ratio of the strategy."""
        excess_returns = self.results_df['strategy_return'] - (self.risk_free_rate / 252)
        sharpe_ratio = (excess_returns.mean() / excess_returns.std()) * np.sqrt(252)
        return sharpe_ratio

    def calculate_total_return(self):
        """Calculate the total return of the strategy."""
        total_return = self.results_df['cumulative_strategy'].iloc[-1] - 1
        return total_return

    def run_backtest(self):
        """Run the backtest and return key metrics."""
        self.calculate_returns()
        sharpe_ratio = self.calculate_sharpe_ratio()
        total_return = self.calculate_total_return()
        
        self.plot_cumulative_returns()
        
        return {
            'Sharpe Ratio': sharpe_ratio,
            'Total Return': total_return
        }

# Example usage
# Assuming news_df and price_df are prepared
# news_df = pd.DataFrame({'title': [...], 'date': [...], 'sentiment_label': [...]})
# price_df = pd.DataFrame({'price': [...], 'date': [...]}) with datetime as index
#
# backtest = NewsTradingBacktest(news_df, price_df, holding_period=3)
# results = backtest.run_backtest()
# print(results)
